# A Copa nova: overwrite dinâmico e a carga das tabelas

## Task 3: carregar a Copa de 2026 sem apagar o histórico e gravar as tabelas

O segundo ponto em aberto do vídeo 5.1: a Copa de 2026, que o `overwrite` trocaria pela tabela inteira.

In [1]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-copa-nova")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
RAW_2026 = "../../data/raw_2026"
OUT = "../../data/out"

spark

A célula abaixo já vem pronta: o código do vídeo 5.1 (`ler_partidas` e o histórico).

In [2]:
# vídeo 5.1: as partidas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

def ler_partidas(pasta):
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )

historico = ler_partidas(RAW)

## Uma Copa nova na saída por Copa

In [7]:
copa_2026 = ler_partidas(RAW_2026)
copa_2026.count()

104

E com o `overwrite` do vídeo 5.1?

In [8]:
copa_2026.write.format("delta").mode("overwrite").partitionBy("ano").save(f"{OUT}/partidas_por_ano")

In [9]:
saida = spark.read.format("delta").load(f"{OUT}/partidas_por_ano")
saida.select("ano").distinct().show()

+----+
| ano|
+----+
|2026|
+----+



O que sobrou? O overwrite **estático** (padrão) troca a tabela inteira. Com `partitionOverwriteMode` em `dynamic`, ele
substitui **só as pastas do lote novo**. Primeiro, o histórico de volta:

In [10]:
historico.write.format("delta").mode("overwrite").partitionBy("ano").save(f"{OUT}/partidas_por_ano")

saida = spark.read.format("delta").load(f"{OUT}/partidas_por_ano")
saida.count()

964

Agora 2026, com o overwrite dinâmico:

In [11]:
(
    copa_2026.write.format("delta").mode("overwrite")
    .option("partitionOverwriteMode", "dynamic")
    .partitionBy("ano")
    .save(f"{OUT}/partidas_por_ano")
)

saida = spark.read.format("delta").load(f"{OUT}/partidas_por_ano")
print(f"{saida.count()} partidas, {saida.select('ano').distinct().count()} Copas")

1068 partidas, 23 Copas


**Rode a célula de novo**, como se o job tivesse rodado outra vez.

Com o modo dinâmico, rodar outra vez não muda o resultado. Isso é **idempotência**.

| Modo | Rodou duas vezes | Histórico |
|---|---|---|
| `append` | duplicou | preservado |
| `overwrite` (estático) | ok | **fora da versão atual** |
| `overwrite` com `dynamic` | ok | **preservado** |

## A carga de verdade

As decisões estão tomadas. Agora, as **tabelas de verdade**: `data/out/partidas` e `data/out/resumo_selecoes`. A mesma
escrita vale para as duas e para cada lote: uma função.

In [12]:
def gravar(df, tabela):
    (
        df.write.format("delta").mode("overwrite")
        .option("partitionOverwriteMode", "dynamic")
        .partitionBy("ano")
        .save(f"{OUT}/{tabela}")
    )

A próxima célula já vem pronta: ela só apaga as duas tabelas, para a carga começar do zero.

In [13]:
import shutil
for tabela in ["partidas", "resumo_selecoes"]:
    shutil.rmtree(f"{OUT}/{tabela}", ignore_errors=True)

Primeiro o histórico, depois 2026. O resumo vem das funções da aula 2:

In [14]:
from wc_pipeline.transformacoes import partidas_por_selecao, resumo_por_selecao

gravar(historico, "partidas")
gravar(resumo_por_selecao(partidas_por_selecao(historico)), "resumo_selecoes")

gravar(copa_2026, "partidas")
gravar(resumo_por_selecao(partidas_por_selecao(copa_2026)), "resumo_selecoes")

In [15]:
partidas = spark.read.format("delta").load(f"{OUT}/partidas")
resumo = spark.read.format("delta").load(f"{OUT}/resumo_selecoes")

print(f"partidas: {partidas.count()} linhas, {partidas.select('ano').distinct().count()} Copas")
print(f"resumo: {resumo.count()} linhas, {resumo.select('ano').distinct().count()} Copas")

partidas: 1068 linhas, 23 Copas
resumo: 537 linhas, 23 Copas


## Levando a carga para o projeto

O `ler_partidas` e o `gravar` vão ser usados de novo, e a carga se repete a cada Copa que o fornecedor mandar. Como no
vídeo 2.4, o código vai para um arquivo do projeto, `src/wc_pipeline/carga.py`:

- fora do notebook não existem as variáveis `spark` e `OUT`, então as funções passam a recebê-las como parâmetro;
- a carga de uma Copa inteira, as partidas e o resumo, vira uma função só: `carregar_copa`.

In [16]:
%%writefile ../../src/wc_pipeline/carga.py
from pyspark.sql import DataFrame, SparkSession

from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao


def ler_partidas(spark: SparkSession, pasta: str) -> DataFrame:
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )


def gravar(df: DataFrame, caminho: str) -> None:
    (
        df.write.format("delta").mode("overwrite")
        .option("partitionOverwriteMode", "dynamic")
        .partitionBy("ano")
        .save(caminho)
    )


def carregar_copa(spark: SparkSession, pasta: str, saida: str) -> None:
    partidas = ler_partidas(spark, pasta)
    gravar(partidas, f"{saida}/partidas")
    gravar(resumo_por_selecao(partidas_por_selecao(partidas)), f"{saida}/resumo_selecoes")

Writing ../../src/wc_pipeline/carga.py


Agora a carga de uma Copa é uma linha. O histórico e 2026 de novo, sem apagar as tabelas:

In [17]:
from wc_pipeline.carga import carregar_copa

carregar_copa(spark, RAW, OUT)
carregar_copa(spark, RAW_2026, OUT)

partidas = spark.read.format("delta").load(f"{OUT}/partidas")
resumo = spark.read.format("delta").load(f"{OUT}/resumo_selecoes")

print(f"partidas: {partidas.count()} linhas, {partidas.select('ano').distinct().count()} Copas")
print(f"resumo: {resumo.count()} linhas, {resumo.select('ano').distinct().count()} Copas")

partidas: 1068 linhas, 23 Copas
resumo: 537 linhas, 23 Copas


Os mesmos números: o código mudou de lugar sem mudar o resultado, e a carga inteira pode rodar de novo sem
duplicar nem apagar nada.

## Resultado da task 3

As tabelas de verdade gravadas, com as 23 Copas, e a carga em `src/wc_pipeline/carga.py`.

## Para consultar depois

- **`replaceWhere`:** no Delta, `.option("replaceWhere", "ano = 2026")` substitui só as linhas que atendem à condição,
  um jeito mais explícito de dizer qual parte da tabela o lote troca.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [18]:
spark.stop()